# 24 — Features derivadas: cantidades relativas y precio relativo

**Idea.** Hasta ahora las cantidades y el precio entran "sueltos". Un árbol corta de a una variable por vez, así que le cuesta armar un **cociente** (vendidas / iniciales) o una **diferencia contra un grupo** (precio contra el precio típico de su categoría). Estas features se las dan hechas.

**Material:** *feature engineering*, `material/Variables Categóricas, Datos Faltantes y Feature Engineering/Copia de AP_Categóricas.ipynb`, sección "Feature Engineering": *"A veces necesitamos generar otra variable que sintetice nueva información a partir de las variables existentes o que capturen distintas características"*. En la notebook de clase, el ejemplo es la superficie **por distrito**: una variable combinada con un grupo, igual que el precio por categoría de D2. Gradient Boosting: `material/Arboles/Árboles.pdf`.

**Pruebas:**
- **D1, cantidades relativas** (determinísticas, afuera del `Pipeline`): `ratio_vendido`, `ratio_disponible`, `vendio_algo`, `diferencia_inicial_disponible`.
- **D2, precio relativo a la categoría** (aprende una mediana → transformer adentro del `Pipeline`): (a) por `category_id`; (b) por `category_id` × moneda.
- **D3, precio relativo al vendedor** (aprende un promedio → transformer adentro del `Pipeline`, excluyendo la propia fila en train).

**Medición:** igual que en la 18 y la 22. Señal sola + aporte marginal sobre el HistGB P7a con duración (0,9412), con comparación pareada (pasa con 5/5, o 4/5 y media > desvío). Lo que pase se combina (con *ablation* si son 2 o más) y se confirma en el promedio final (CatBoost + HistGB, 0,9420) con las semillas 42 y 7.

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      agregar_logs, CapeoPorMoneda, PrediccionOOF, armar_modelo_texto,
                      armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 200)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
crudo_test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)

# Referencia por fold de la 18 (semilla 42): HistGB P7a + duración y modelo final (igual que en la 22 y la 23)
oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
def acc_por_fold(prob, folds):
    return pd.Series((np.asarray(prob) >= 0.5).astype(int) == y.to_numpy()).groupby(folds).mean().to_numpy()
acc_ref = acc_por_fold(oof_ref["histgb"], oof_ref["fold"])
acc_final_42 = acc_por_fold(0.5 * oof_ref["catboost"] + 0.5 * oof_ref["histgb"], oof_ref["fold"])
print("HistGB P7a + duración:", acc_ref.round(4), acc_ref.mean().round(4))
print("Modelo final (s. 42): ", acc_final_42.round(4), acc_final_42.mean().round(4))

HistGB P7a + duración: [0.9428 0.9393 0.9426 0.9394 0.9422] 0.9412
Modelo final (s. 42):  [0.9431 0.9389 0.9424 0.942  0.9436] 0.942


## D1 — Cantidades relativas

**Chequeo previo (confirmado):**
- `initial_quantity` **nunca es 0**, ni en train ni en test, así que los cocientes siempre existen. Igual se dejan en NaN si el denominador es 0 (el HistGB maneja los NaN solo) para que la función no se rompa con otros datos.
- `sold_quantity` **supera** a `initial_quantity` en el 6% de las filas. `ratio_vendido` puede pasar de 1. **Suposición (no verificada):** `sold_quantity` acumula ventas de publicaciones anteriores (republicadas) y `initial_quantity` es solo de esta.
- `initial − available` no siempre es igual a `sold_quantity` (lo es en el 85% de las filas), así que `vendio_algo` y `diferencia_inicial_disponible` no son lo mismo que `sold_quantity`.

Las features: `ratio_vendido` = sold / initial; `ratio_disponible` = available / initial; `vendio_algo` = 1 si initial − available > 0; `diferencia_inicial_disponible` = initial − available. Son determinísticas (dependen solo de la propia fila, no aprenden nada), así que pueden ir afuera del `Pipeline`, como `duracion_dias`.

In [2]:
def agregar_cantidades_relativas(df):
    df = df.copy()
    inicial = df["initial_quantity"].where(df["initial_quantity"] > 0)   # 0 -> NaN (no pasa en estos datos)
    df["ratio_vendido"] = df["sold_quantity"] / inicial
    df["ratio_disponible"] = df["available_quantity"] / inicial
    df["diferencia_inicial_disponible"] = df["initial_quantity"] - df["available_quantity"]
    df["vendio_algo"] = (df["diferencia_inicial_disponible"] > 0).astype(int)
    return df


D1 = ["ratio_vendido", "ratio_disponible", "vendio_algo", "diferencia_inicial_disponible"]
train = agregar_cantidades_relativas(train)
test_d1 = agregar_cantidades_relativas(crudo_test)
print("initial_quantity == 0 -> train:", (crudo["initial_quantity"] == 0).sum(), "| test:", (crudo_test["initial_quantity"] == 0).sum())
print("sold > initial -> train:", (crudo["sold_quantity"] > crudo["initial_quantity"]).mean().round(4),
      "| test:", (crudo_test["sold_quantity"] > crudo_test["initial_quantity"]).mean().round(4))


def tabla_tramos(serie_train, serie_test, cortes, etiquetas):
    tr = pd.cut(serie_train, cortes, labels=etiquetas, include_lowest=True)
    te = pd.cut(serie_test, cortes, labels=etiquetas, include_lowest=True)
    return pd.DataFrame({"filas_train": tr.value_counts().reindex(etiquetas),
                         "pct_train": tr.value_counts(normalize=True).reindex(etiquetas),
                         "pct_test": te.value_counts(normalize=True).reindex(etiquetas),
                         "pct_new": y.groupby(tr, observed=False).mean().reindex(etiquetas)}).round(3)


print("\n=== ratio_vendido ===")
print(tabla_tramos(train["ratio_vendido"], test_d1["ratio_vendido"], [-0.001, 0, 0.25, 0.5, 1, 2, np.inf],
                   ["0", "0-0,25", "0,25-0,5", "0,5-1", "1-2", "> 2"]))
print("\n=== ratio_disponible ===")
print(tabla_tramos(train["ratio_disponible"], test_d1["ratio_disponible"], [-0.001, 0.5, 0.9, 0.999, 1],
                   ["< 0,5", "0,5-0,9", "0,9-1", "1 (no vendió)"]))
print("\n=== diferencia_inicial_disponible ===")
print(tabla_tramos(train["diferencia_inicial_disponible"], test_d1["diferencia_inicial_disponible"], [-0.001, 0, 1, 5, np.inf],
                   ["0", "1", "2-5", "> 5"]))
print("\n=== cruce: vendio_algo × sold_quantity > 0 (% new y filas) ===")
cruce = [train["vendio_algo"], (train["sold_quantity"] > 0).rename("sold_mayor_0")]
print(pd.crosstab(*cruce, values=y, aggfunc="mean").round(3))
print(pd.crosstab(*cruce))

initial_quantity == 0 -> train: 0 | test: 0
sold > initial -> train: 0.0604 | test: 0.0593

=== ratio_vendido ===
               filas_train  pct_train  pct_test  pct_new
ratio_vendido                                           
0                    58098      0.830     0.833    0.465
0-0,25                2667      0.038     0.038    0.965
0,25-0,5              1386      0.020     0.019    0.952
0,5-1                 3622      0.052     0.051    0.741
1-2                   1601      0.023     0.022    0.913
> 2                   2626      0.038     0.037    0.972

=== ratio_disponible ===
                  filas_train  pct_train  pct_test  pct_new
ratio_disponible                                           
< 0,5                     863      0.012     0.013    0.968
0,5-0,9                  1570      0.022     0.022    0.961
0,9-1                     698      0.010     0.009    0.983
1 (no vendió)           66869      0.955     0.956    0.517

=== diferencia_inicial_disponible ===
     

**Lectura de D1.**
- **Las cantidades separan muy bien, pero en pocas filas:** cualquier señal de venta (ratio vendido > 0, ratio disponible < 1, diferencia > 0) da 74-99% de `new`, contra 46-52% en el resto. El 83% de las filas no vendió nada.
- `ratio_vendido` > 2 (3,8% de las filas, 97% `new`): son las filas donde `sold_quantity` supera a `initial_quantity`. Train y test tienen las mismas proporciones en todos los tramos.
- **Redundancia (confirmado en el cruce):** `vendio_algo` = 1 es un **subconjunto** de `sold_quantity` > 0. Las 3.161 filas con `vendio_algo` = 1 tienen todas `sold_quantity` > 0 (96,8% `new`, contra 86,1% del resto de las que vendieron). `sold_quantity` ya está en el modelo, así que lo nuevo es solo esa diferencia de 11 puntos en ~4,5% de las filas.
- **Por qué la señal sola da 0,537 en tres de ellas:** ese es el % de la clase mayoritaria (`new`). Con una sola de esas features, el árbol predice `new` en los dos lados del corte (96% de un lado y 52% del otro), así que no cambia ninguna predicción respecto de "todo `new`". Aunque la feature sí tiene información, la accuracy de una sola feature no la muestra.

## D2 — Precio relativo a la categoría

`precio_rel_categoria = log_price − mediana de log_price de su category_id`. Positivo = más caro que lo típico de su categoría.

**¿Por qué va adentro del `Pipeline` si no usa el target?** Porque **aprende algo de los datos**: la mediana de cada categoría. Si se calculara con todo el train antes de la CV, las filas del fold de validación ayudarían a calcular la mediana de su propia categoría, cosa que con el test real no pasa (el modelo se entrena sin ver el test). La validación mediría entonces una situación un poco más favorable que la real. Adentro del `Pipeline`, las medianas se calculan solo con el train de cada fold, y la validación queda en las mismas condiciones que el test. Es la regla metodológica del TP: todo lo que aprende de los datos va adentro, como el capeo por moneda (`CapeoPorMoneda`), que tampoco usa el target.

**Categorías chicas:** con menos de `minimo` = 5 publicaciones en el train del fold, la mediana es poco confiable y se usa la mediana global. Lo mismo para categorías no vistas.

**Variantes:** (a) grupo = `category_id`; (b) grupo = `category_id` × `currency_id`. Los precios en USD (399 filas) y en ARS no son comparables: una moto en USD tiene un log_price mucho menor que en pesos.

Un detalle: en el train, cada fila participa de la mediana de su categoría. Con el mínimo de 5 filas y la mediana (robusta: una fila no la mueve mucho), el efecto es chico. No se excluye la propia fila (eso se hace en D3, donde es más importante).

In [3]:
class PrecioRelativoGrupo(BaseEstimator, TransformerMixin):
    # log_price − mediana de log_price del grupo (aprendida en fit). Grupos chicos o no vistos -> mediana global.
    def __init__(self, columnas_grupo, minimo=5):
        self.columnas_grupo = columnas_grupo
        self.minimo = minimo

    def _grupo(self, X):
        return X[self.columnas_grupo].astype(str).agg("_".join, axis=1)

    def fit(self, X, y=None):
        g = X.groupby(self._grupo(X))["log_price"]
        tamanio, mediana = g.size(), g.median()
        self.medianas_ = mediana[tamanio >= self.minimo]
        self.mediana_global_ = X["log_price"].median()
        return self

    def transform(self, X):
        referencia = self._grupo(X).map(self.medianas_).fillna(self.mediana_global_)
        return (X["log_price"] - referencia).to_numpy().reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["precio_rel_grupo"])

## D3 — Precio relativo al vendedor

`precio_rel_vendedor = log_price − promedio de log_price de las OTRAS publicaciones del mismo vendedor`.

**El problema de incluir la propia fila.** En train, el promedio del vendedor se calcula con filas que incluyen a la propia; en validación/test, la fila nunca participó del promedio. Si no se corrige, la feature se comporta distinto en train que en test, y el modelo aprende reglas que después no valen.

**Ejemplo de 3 filas.** En el train de un fold, el vendedor A tiene 3 publicaciones con log_price 2, 4 y 6; el vendedor B tiene 1, con log_price 3.

| Fila | Vendedor | log_price | Incluyendo la propia fila | **Excluyéndola (lo que se usa)** |
|---|---|---|---|---|
| 1 | A | 2 | 2 − 4 = −2 | 2 − (4+6)/2 = **−3** |
| 2 | A | 4 | 4 − 4 = 0 | 4 − (2+6)/2 = **0** |
| 3 | A | 6 | 6 − 4 = +2 | 6 − (2+4)/2 = **+3** |
| 4 | B | 3 | 3 − 3 = **0 siempre** | sin otras publicaciones → **NaN** |

Y en validación/test (se usa el promedio de **todas** las filas del vendedor en el train, porque la fila nueva no está ahí):
- una publicación de A con log_price 5 → 5 − 4 = +1;
- una publicación de un vendedor C que no está en el train → **NaN**.

**Qué arregla la exclusión:**
1. Sin exclusión, todos los vendedores con 1 publicación dan **0 exacto** en train (19.926 filas), pero en test un vendedor nuevo da NaN: el modelo aprendería algo sobre "0" que en test nunca aparece. Con la exclusión, los dos casos dan NaN, que significa lo mismo en los dos lados: "no hay otras publicaciones del vendedor para comparar".
2. Sin exclusión, la propia fila "tira" el promedio hacia ella y achica la diferencia (−2 contra −3), sobre todo en vendedores chicos. En test no pasa.

Es el mismo *leave-one-out* que `VendedorEncoder` usa para `seller_pct_new`, pero sobre el precio (sin target). El HistGB maneja los NaN solo.

In [4]:
class PrecioRelativoVendedor(BaseEstimator, TransformerMixin):
    # log_price − promedio de log_price de las OTRAS publicaciones del vendedor (sin la propia fila en train)
    def fit(self, X, y=None):
        g = X.groupby("seller_id")["log_price"]
        self.suma_, self.conteo_ = g.sum(), g.size()
        return self

    def transform(self, X):
        # Validación / test: la fila no participó del fit -> promedio de todas las del vendedor; no visto -> NaN
        promedio = X["seller_id"].map(self.suma_ / self.conteo_)
        return (X["log_price"] - promedio).to_numpy().reshape(-1, 1)

    def fit_transform(self, X, y=None):
        # Train: se resta la propia fila de la suma y del conteo; sin otras publicaciones -> NaN
        self.fit(X)
        suma, conteo = X["seller_id"].map(self.suma_), X["seller_id"].map(self.conteo_)
        promedio_otras = ((suma - X["log_price"]) / (conteo - 1)).where(conteo > 1)
        return (X["log_price"] - promedio_otras).to_numpy().reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["precio_rel_vendedor"])


# Verificación con el ejemplo de 3 filas
ejemplo = pd.DataFrame({"seller_id": ["A", "A", "A", "B"], "log_price": [2.0, 4.0, 6.0, 3.0]})
t = PrecioRelativoVendedor()
print("fit_transform (train):", t.fit_transform(ejemplo).ravel())
print("transform (test, A con 5 y C con 3):", t.transform(pd.DataFrame({"seller_id": ["A", "C"], "log_price": [5.0, 3.0]})).ravel())

fit_transform (train): [-3.  0.  3. nan]
transform (test, A con 5 y C con 3): [ 1. nan]


## Medición

Cada candidata se mide como en la 18 y la 22:
- **señal sola:** `HistGradientBoostingClassifier` con solo esa feature (para D2/D3, con capeo + logs + el transformer adelante);
- **aporte marginal:** `armar_pipeline_histgb_combos()` (HistGB P7a con duración, modelo de texto cacheado) + la feature, con comparación pareada contra 0,9412.

In [5]:
pasos_previos = [("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
                 ("logs", FunctionTransformer(agregar_logs))]
# Cada candidata: lista de (nombre, transformer, columnas) que se agregan al ColumnTransformer
candidatas = {
    "D1a ratio_vendido": [("d1", "passthrough", ["ratio_vendido"])],
    "D1b ratio_disponible": [("d1", "passthrough", ["ratio_disponible"])],
    "D1c vendio_algo": [("d1", "passthrough", ["vendio_algo"])],
    "D1d diferencia_inicial_disponible": [("d1", "passthrough", ["diferencia_inicial_disponible"])],
    "D1 (las 4 juntas)": [("d1", "passthrough", D1)],
    "D2a precio_rel_categoria": [("d2", PrecioRelativoGrupo(["category_id"]), ["category_id", "log_price"])],
    "D2b precio_rel_categoria_moneda": [("d2", PrecioRelativoGrupo(["category_id", "currency_id"]),
                                         ["category_id", "currency_id", "log_price"])],
    "D3 precio_rel_vendedor": [("d3", PrecioRelativoVendedor(), ["seller_id", "log_price"])],
}


def con_texto_cacheado(pipeline):
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def p7a_con(extras):
    pipeline = con_texto_cacheado(armar_pipeline_histgb_combos())
    pipeline.named_steps["preprocesador"].transformers.extend(extras)
    return pipeline


def solo(extras):
    return Pipeline(pasos_previos + [("preprocesador", ColumnTransformer(extras, sparse_threshold=0)),
                                     ("modelo", HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None,
                                                                               max_iter=200, random_state=SEED))])


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


def medir(nombre, extras, con_sola=True):
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sola = (cross_validate(solo(extras), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"].mean()
                if con_sola else np.nan)
        marginal = cross_validate(p7a_con(extras), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
    fila = {"candidata": nombre, "señal_sola": sola, "con_P7a": marginal.mean(), **comparar(marginal, acc_ref),
            "minutos": (time.time() - inicio) / 60}
    print(f"{nombre:36s} sola {sola:.4f} | con P7a {marginal.mean():.4f} | dif {fila['dif_media']:+.4f} "
          f"({fila['folds_gana']}/5){' PASA' if fila['pasa'] else ''} | {fila['minutos']:.1f} min")
    return fila


filas = [medir(nombre, extras) for nombre, extras in candidatas.items()]
tabla = pd.DataFrame(filas).set_index("candidata")
tabla.round(4)

D1a ratio_vendido                    sola 0.5955 | con P7a 0.9411 | dif -0.0001 (3/5) | 0.8 min


D1b ratio_disponible                 sola 0.5370 | con P7a 0.9412 | dif +0.0000 (2/5) | 0.7 min


D1c vendio_algo                      sola 0.5370 | con P7a 0.9409 | dif -0.0003 (0/5) | 0.7 min


D1d diferencia_inicial_disponible    sola 0.5370 | con P7a 0.9406 | dif -0.0007 (0/5) | 0.7 min


D1 (las 4 juntas)                    sola 0.5955 | con P7a 0.9412 | dif +0.0000 (2/5) | 0.7 min


D2a precio_rel_categoria             sola 0.5998 | con P7a 0.9410 | dif -0.0003 (2/5) | 0.9 min


D2b precio_rel_categoria_moneda      sola 0.6012 | con P7a 0.9408 | dif -0.0005 (1/5) | 0.8 min


D3 precio_rel_vendedor               sola 0.5583 | con P7a 0.9412 | dif -0.0000 (3/5) | 0.8 min


,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos
candidata,,,,,,,
D1a ratio_vendido,0.5955,0.9411,-0.0001,0.0008,3,False,0.8219
D1b ratio_disponible,0.5370,0.9412,0.0000,0.0005,2,False,0.6952
D1c vendio_algo,0.5370,0.9409,-0.0003,0.0004,0,False,0.7067
D1d diferencia_inicial_disponible,0.5370,0.9406,-0.0007,0.0004,0,False,0.6766
D1 (las 4 juntas),0.5955,0.9412,0.0000,0.0011,2,False,0.7375
D2a precio_rel_categoria,0.5998,0.9410,-0.0003,0.0005,2,False,0.9014
D2b precio_rel_categoria_moneda,0.6012,0.9408,-0.0005,0.0011,1,False,0.8328
D3 precio_rel_vendedor,0.5583,0.9412,-0.0000,0.0016,3,False,0.7546


## Combinación y *ablation* de las que pasan

Si pasan 2 o más candidatas, se miden juntas y sacando de a una (*ablation*, como en `11b_marginales`). D1 se cuenta como bloque solo si pasó "D1 (las 4 juntas)" y ninguna suelta; D2a y D2b son variantes de lo mismo, así que entra la mejor de las que pasen.

In [6]:
pasan = [n for n in tabla.index if tabla.loc[n, "pasa"]]
if any(n.startswith("D1") and n != "D1 (las 4 juntas)" for n in pasan):
    pasan = [n for n in pasan if n != "D1 (las 4 juntas)"]
d2 = [n for n in pasan if n.startswith("D2")]
if len(d2) > 1:
    pasan = [n for n in pasan if n not in d2] + [tabla.loc[d2, "dif_media"].idxmax()]
print("Pasan:", pasan)


def juntar(nombres):
    extras = []
    for k, n in enumerate(nombres):
        extras += [(f"{nom}_{k}", t, cols) for nom, t, cols in candidatas[n]]
    return extras


filas_comb = []
if len(pasan) >= 2:
    filas_comb.append(medir("Juntas: " + " + ".join(pasan), juntar(pasan), con_sola=False))
    for sacada in pasan:
        filas_comb.append(medir(f"Juntas sin {sacada}", juntar([n for n in pasan if n != sacada]), con_sola=False))
    pd.DataFrame(filas_comb).set_index("candidata").round(4)
else:
    print("Menos de 2 candidatas pasan: no hay combinación ni ablation.")

Pasan: []
Menos de 2 candidatas pasan: no hay combinación ni ablation.


## Confirmación en el promedio final (CatBoost + HistGB + duración), semillas 42 y 7

Solo si algo pasó. La configuración ganadora (`ganadora`: todas las que pasan, o la mejor combinación del *ablation*) se suma **a los dos modelos**:
- al HistGB P7a, como arriba;
- a CatBoost, **al final** del `ColumnTransformer` (después de las categóricas crudas), así `INDICES_CAT` no cambia.

Promedio 50/50, comparación pareada contra el modelo final: semilla 42 contra las OOF guardadas en la 18; semilla 7 contra el modelo final recalculado con esa semilla.

In [7]:
filas_comb_df = pd.DataFrame(filas_comb).set_index("candidata") if filas_comb else None
if not pasan:
    ganadora = []
    print("Nada pasó sobre el HistGB P7a: no hay confirmación.")
elif len(pasan) == 1 or filas_comb_df.loc["Juntas: " + " + ".join(pasan), "dif_media"] >= filas_comb_df["dif_media"].max():
    ganadora = pasan
else:
    ganadora = [n for n in pasan if n != filas_comb_df["dif_media"].idxmax().replace("Juntas sin ", "")]
print("Configuración a confirmar:", ganadora)


def promedio_final(extras, semilla):
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
    folds_s = np.empty(len(train), dtype=int)
    for k, (_, v) in enumerate(cv_s.split(train, y)):
        folds_s[v] = k + 1
    cat = con_texto_cacheado(armar_pipeline_final())
    cat.named_steps["preprocesador"].transformers.extend(extras)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        p_cat = cross_val_predict(cat, train, y, cv=cv_s, method="predict_proba", n_jobs=5,
                                  params={"modelo__cat_features": INDICES_CAT})[:, 1]
        p_hgb = cross_val_predict(p7a_con(extras), train, y, cv=cv_s, method="predict_proba", n_jobs=5)[:, 1]
    return acc_por_fold(p_cat, folds_s), acc_por_fold(p_hgb, folds_s), acc_por_fold(0.5 * p_cat + 0.5 * p_hgb, folds_s)


confirmacion = {}
if ganadora:
    for semilla in [42, 7]:
        inicio = time.time()
        _, _, acc_nuevo = promedio_final(juntar(ganadora), semilla)
        acc_base = acc_final_42 if semilla == 42 else promedio_final([], 7)[2]
        confirmacion[semilla] = {"modelo final": acc_base.mean(), "con la nueva": acc_nuevo.mean(),
                                 **comparar(acc_nuevo, acc_base), "minutos": (time.time() - inicio) / 60}
        print(semilla, confirmacion[semilla])
pd.DataFrame(confirmacion).round(4)

Nada pasó sobre el HistGB P7a: no hay confirmación.
Configuración a confirmar: []


""


## Conclusión

**Nada pasa.** Referencia: HistGB P7a con duración = 0,9412.

| Candidata | Señal sola | Con P7a | Diferencia (folds) |
|---|---|---|---|
| D1a `ratio_vendido` | 0,5955 | 0,9411 | −0,0001 (3/5) |
| D1b `ratio_disponible` | 0,5370 | 0,9412 | +0,0000 (2/5) |
| D1c `vendio_algo` | 0,5370 | 0,9409 | −0,0003 (0/5) |
| D1d `diferencia_inicial_disponible` | 0,5370 | 0,9406 | −0,0007 (0/5) |
| D1 (las 4 juntas) | 0,5955 | 0,9412 | +0,0000 (2/5) |
| D2a precio relativo a la categoría | 0,5998 | 0,9410 | −0,0003 (2/5) |
| D2b precio relativo a categoría × moneda | 0,6012 | 0,9408 | −0,0005 (1/5) |
| D3 precio relativo al vendedor (sin la propia fila) | 0,5583 | 0,9412 | −0,0000 (3/5) |

~0,8 min por candidata. No hubo combinación, *ablation* ni confirmación en el promedio final (no pasó ninguna).

- **D1:** lo que dicen las cantidades relativas ya lo dice `sold_quantity` (que está en el modelo). `vendio_algo` es un subconjunto de `sold_quantity` > 0.
- **D2:** el precio relativo a la categoría tiene algo de señal sola (0,60), pero el HistGB ya tiene `log_price` + `category_id` codificada + la combinación categoría × bin de precio (`c_cat_price`), y con eso arma lo mismo. Separar por moneda no ayuda (USD son 399 filas).
- **D3:** el precio relativo al vendedor no suma sobre el conteo y el precio promedio del vendedor que el HistGB ya tiene (`VendedorEncoder` sin target) más las combinaciones vendedor × bin de precio (`c_seller_price`).
- **Decisión:** no se cambia nada. Modelo final = P7c + duración (0,9420). Sin cambios en `features.py` y sin submission.